In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

In [4]:
ruta_base = r"C:\Users\Lenovo\Documents\GitHub\Proyecto_Analisis_Datos_2026\data"
carpeta_origen = Path('data/raw')
carpeta_origen = Path(ruta_base) / 'raw'
carpeta_destino = Path(ruta_base) / 'processed'

carpeta_destino.mkdir(parents=True, exist_ok=True)

In [6]:
def limpiar_datos(df):

    df_limpio = df.drop_duplicates()
    df_limpio = df_limpio.dropna()
    return df_limpio
for archivo in carpeta_origen.iterdir():
    if archivo.is_file():
        print(f"Procesando: {archivo.name}...")
        
        try:
            if archivo.suffix == '.csv':
                df = pd.read_csv(archivo)
                df_limpio = limpiar_datos(df)
                
                ruta_salida = carpeta_destino / f"limpio_{archivo.name}"
                df_limpio.to_csv(ruta_salida, index=False)
                print(f"  -> Guardado como CSV: {ruta_salida}")

            elif archivo.suffix == '.json':
                df = pd.read_json(archivo)
                df_limpio = limpiar_datos(df)
                
                ruta_salida = carpeta_destino / f"limpio_{archivo.name}"
                df_limpio.to_json(ruta_salida, orient='records', indent=4)
                print(f"  -> Guardado como JSON: {ruta_salida}")

            elif archivo.suffix in ['.xlsx', '.xls']:
                df = pd.read_excel(archivo)
                df_limpio = limpiar_datos(df)
                
                ruta_salida = carpeta_destino / f"limpio_{archivo.name}"
                df_limpio.to_excel(ruta_salida, index=False)
                print(f"  -> Guardado como EXCEL: {ruta_salida}")

            elif archivo.suffix in ['.sqlite', '.db']:
                conexion_origen = sqlite3.connect(archivo)
                ruta_salida = carpeta_destino / f"limpio_{archivo.name}"
                conexion_destino = sqlite3.connect(ruta_salida)
                
                query_tablas = "SELECT name FROM sqlite_master WHERE type='table';"
                tablas = pd.read_sql_query(query_tablas, conexion_origen)['name'].tolist()
                
                for tabla in tablas:
                    df = pd.read_sql_query(f"SELECT * FROM {tabla}", conexion_origen)
                    df_limpio = limpiar_datos(df)
                    df_limpio.to_sql(tabla, conexion_destino, if_exists='replace', index=False)
                    
                conexion_origen.close()
                conexion_destino.close()
                print(f"  -> Guardado como SQLite: {ruta_salida}")

            else:
                print(f"  -> Formato no soportado: {archivo.suffix}, saltando...")
                
        except Exception as e:
            print(f"  -> Error procesando {archivo.name}: {e}")
            
print("\n¡Procesamiento por lotes finalizado con éxito!")

Procesando: base-de-datos-proyectos-de-economias-comunitarias.csv...
  -> Error procesando base-de-datos-proyectos-de-economias-comunitarias.csv: 'utf-8' codec can't decode byte 0xd3 in position 15: invalid continuation byte
Procesando: fuente_economia_desempleo_ecuador.sqlite...
  -> Guardado como SQLite: C:\Users\Lenovo\Documents\GitHub\Proyecto_Analisis_Datos_2026\data\processed\limpio_fuente_economia_desempleo_ecuador.sqlite
Procesando: ieps_proyecto-inversion_2021noviembre-.json...
  -> Guardado como JSON: C:\Users\Lenovo\Documents\GitHub\Proyecto_Analisis_Datos_2026\data\processed\limpio_ieps_proyecto-inversion_2021noviembre-.json
Procesando: inec_enemdu_consumidor_2021_septiembre.csv...
  -> Guardado como CSV: C:\Users\Lenovo\Documents\GitHub\Proyecto_Analisis_Datos_2026\data\processed\limpio_inec_enemdu_consumidor_2021_septiembre.csv
Procesando: mies_bonos_pensiones_2026_abril_csv.zip...
  -> Formato no soportado: .zip, saltando...
Procesando: proyectos_ieps.json...
  -> Guarda